# Part 2: RFM Customer Segmentation and Strategy Analysis
**Course:** AIML Capstone Project  
**Task:** Build traditional RFM features, map heuristic segment logic, and analyze group metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Environment initialized.")

## 1. Data Aggregation & RFM Matrix Building

In [ ]:
orders = pd.read_csv("orders.csv")
tickets = pd.read_csv("support_tickets.csv")

orders['order_date'] = pd.to_datetime(orders['order_date'])
snapshot_date = pd.to_datetime("2025-09-30")

rfm = orders.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'gross_amount': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']
ticket_counts = tickets.groupby('customer_id').size().reset_index(name='complaints_count')

rfm_final = pd.merge(rfm, ticket_counts, on='customer_id', how='left')
rfm_final['complaints_count'] = rfm_final['complaints_count'].fillna(0)

print(rfm_final.head())

## 2. Applying Segmentation Boundary Rules

In [ ]:
def assign_student_segment(row):
    if row['recency'] > 180:
        return 'Dormant Customers'
    elif row['recency'] <= 60 and row['monetary'] > 1000 and row['complaints_count'] >= 3:
        return 'High-Value But Unhappy'
    elif row['recency'] <= 45 and row['frequency'] >= 5:
        return 'Champions'
    elif row['frequency'] <= 2 and row['recency'] <= 90:
        return 'At-Risk Customers'
    else:
        return 'Loyal Customers'

rfm_final['segment_name'] = rfm_final.apply(assign_student_segment, axis=1)
print(rfm_final['segment_name'].value_counts())

## 3. Data Visualization (6 Plots For Cohort Analysis)
### Chart 1: Segment Size Breakdown

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.countplot(data=rfm_final, y='segment_name', palette='Set2', order=rfm_final['segment_name'].value_counts().index)
plt.title('Customer Count Across Derived RFM Segments')
plt.xlabel('Volume of Customers')
plt.ylabel('Segment Identifier')
plt.savefig('segment_counts_plot.png', bbox_inches='tight')
plt.show()

### Chart 2: Recency Distribution across Clusters

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=rfm_final, x='recency', y='segment_name', palette='coolwarm')
plt.title('Recency Matrix Variance Across Segments')
plt.xlabel('Days Since Last Transaction')
plt.ylabel('Segment Name')
plt.savefig('recency_variance_plot.png', bbox_inches='tight')
plt.show()

### Chart 3: Order Frequency Characteristics

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(data=rfm_final, x='segment_name', y='frequency', palette='magma', errorbar=None)
plt.title('Mean Order Frequency per Customer Segment')
plt.xlabel('Segment Name')
plt.ylabel('Average Order Count')
plt.xticks(rotation=20)
plt.savefig('frequency_means_plot.png', bbox_inches='tight')
plt.show()

### Chart 4: Monetary Spends by Cohort Group

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=rfm_final, x='monetary', y='segment_name', palette='viridis')
plt.title('Total Revenue Monetary Footprint Distribution')
plt.xlabel('Gross Monetary Expenditure ($)')
plt.ylabel('Segment Name')
plt.savefig('monetary_footprint_plot.png', bbox_inches='tight')
plt.show()

### Chart 5: Complaint Densities Across Operational Tiers

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(data=rfm_final, x='segment_name', y='complaints_count', palette='rocket', errorbar=None)
plt.title('Average Logged Complaints Density by Segment Group')
plt.xlabel('Segment Name')
plt.ylabel('Mean Support Tickets Filed')
plt.xticks(rotation=20)
plt.savefig('complaint_densities_plot.png', bbox_inches='tight')
plt.show()

### Chart 6: Recency vs Monetary Value Cross Plot

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=rfm_final, x='recency', y='monetary', hue='segment_name', alpha=0.6, palette='tab10')
plt.title('Recency vs Total Monetary Spend Mapping Scatter')
plt.xlabel('Days Since Last Order')
plt.ylabel('Total Financial Spend ($)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig('recency_monetary_scatter.png', bbox_inches='tight')
plt.show()